In [4]:
print("Job Searching Agent - Experiments")

name = "Job Search Agent"
version = "0.1"

print(f"Agent: {name}")
print(f"Version: {version}")

Job Searching Agent - Experiments
Agent: Job Search Agent
Version: 0.1


In [14]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# 1. Load variables from .env
load_dotenv()

# 2. Initialize the OpenRouter client
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

# 3. Test a fast completion call
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "Hello! Confirm you are ready for the Job Searching Agent project."}
    ]
)

print(response.choices[0].message.content)

Hello! I’m all set and ready to dive into the Job Searching Agent project. Let’s get started!


In [6]:
%pip install --upgrade\
langchain langchain-openai langgraph langsmith openai \
    python-docx pdfplumber ipython pydantic typing_extensions

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\MANIKANTA\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [15]:
import os
import subprocess
import docx
import pdfplumber
from langchain_core.tools import tool

In [26]:
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "your_langsmith_token_here"
os.environ["LANGSMITH_PROJECT"] = "jobagent"

In [17]:
@tool
def extract_cv_text(file_path: str) -> str:
    """Extracts the text content from a CV in PDF,DOCX or DOC format. The CV should the same folder that this notebook is in.
    Args:
    file_path(str):The local file path to the CV document.
    Returns:
    str: The extracted plain text from the CV, or an error message if the format is unsupported or cannot be read.
    """
    ext=os.path.splitext(file_path)[-1].lower()
    if ".docx" in ext:
        try:
            doc = docx.Document(file_path)
            text=[para.text for para in doc.paragraphs]
            return '\n'.join(text)
        except Exception as e:
            return f"Error reading the .doc file: {e}"
    elif".pdf" in ext:
        try:
            text=[]
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    page_text= page.extract_text()
                    if page_text:
                        text.append(page_text)
            return '\n'.join(text)
        except Exception as e:
            return f"Error reading the .pdf file: {e}"

    elif ".doc" in ext:
        try:
            temp_docx=file_path+ ".temp.docx"
            subprocess.run(['soffice','--headless','--convert-to','docx','--outdir',os.path.dirname(file_path),file_path],check=True)

            doc =docx.Document(os.path.splitext(file_path)[0]+".docx")
            text=[para.text for para in doc.paragraphs]

            os.remove(os.path.splitext(file_path)[0]+".docx")
            return '\n'.join(text)
        except Exception as e:
            return f"Error reading .doc file: {e}"

    else:
        return "Unsupported file format! please use .pdf or docx or .doc"

In [18]:
file_name = r"C:\Users\MANIKANTA\OneDrive\Desktop\CV-Template.pdf"

text = extract_cv_text.invoke({
    "file_path": file_name
})

print(type(text))
print(text)

<class 'str'>
Fred Smith, CFA
fsmith@email.com
+ 44 (0) 7777 777 777
EDUCATION
2010 – 2014 CFA Institute
Chartered Financial Analyst
2007 – 2010 ABC University, UK
BSc (Mathematics), First Class Honours
BUSINESS EXPERIENCE
2012 - 2015 XYZ Capital Ltd, London, UK
Private equity firm with £100m AUM
Senior Associate
 Reviewed and led c. 100 equity and mezzanine deals/investments of which 10 were completed
(total deal size $1.5bn+) and 2, in manufacturing and consumer goods, were exited
 Managed projects and conducted financial and commercial due diligence for target companies
across a number of industry sectors and product offerings covering both equity and debt
 Dissuaded team from pursuing two fraudulent manufacturing investments of around $10m each
after initiating extensive additional due diligence, also saving significant deal fees
 Created a complex two-phase LBO model associated with significant acquisition of bolt-on
company for a multinational insurance platform
 Led a team 

In [19]:
job_link="https://www.naukri.com/"

In [20]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import MessagesState

from openai import OpenAI
import os


# ==========================================
# OpenRouter client
# ==========================================

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)


# ==========================================
# Job Posting Tool
# ==========================================

@tool
def job_posting_tool(job_link: str) -> str:
    """
    Extract structured information from a job posting at the provided URL.

    Args:
        job_link: The URL of the job posting.

    Returns:
        A structured summary of the job posting's key details.
    """

    completion = client.chat.completions.create(
        model="openai/gpt-oss-20b",

        tools=[
            {
                "type": "openrouter:web_search",
                "parameters": {
                    "engine": "exa",
                    "search_context_size": "medium"
                }
            }
        ],

        messages=[
            {
                "role": "system",
                "content": """
You are a helpful tool that visits a provided job posting
and carefully analyzes its contents.

Read the job posting thoroughly and summarize the important
information, including:

- Job title
- Company name
- Location
- Employment type
- Salary or compensation, if available
- Required qualifications and skills
- Main responsibilities
- Benefits
- Application instructions
- Posting date, if available

Format:

- Give the response as a clear, structured bullet-point list.
- Use factual information from the job posting.
- Keep the summary concise.
- Do not invent information.
- If the job posting cannot be accessed, say that it is unavailable.

Do's:

- Make sure every extracted detail is accurate.
- Keep descriptions concise and professional.
- Use consistent formatting.

Don'ts:

- Don't add guesses or speculation.
- Don't add personal opinions.
- Don't add information that isn't present in the posting.
"""
            },
            {
                "role": "user",
                "content": f"""
Visit this job posting and extract details:

{job_link}
"""
            }
        ]
    )

    return (
            completion.choices[0].message.content
            or "No job information was returned."
    )


# ==========================================
# Your LangChain tools
# ==========================================

tools = [
    job_posting_tool,
    extract_cv_text
]


# ==========================================
# LangGraph ToolNode
# ==========================================

tool_node = ToolNode(tools=tools)


# ==========================================
# OpenRouter LangChain LLM
# ==========================================

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)


# IMPORTANT: bind AFTER defining tools

llm_with_tools = llm.bind_tools(tools)


# ==========================================
# System Message
# ==========================================

sys_msg = SystemMessage(content="""
You are an expert career assistant that helps the user
with questions related to jobs, careers, and applications.

Your key capabilities:

- You have access to the user's CV and can read its contents
  using the `extract_cv_text` tool.
- You can look up and extract details from job postings using
  the `job_posting_tool`.
- You can compare the user's CV against one or more job
  postings to determine suitability and provide tailored advice.
- You can suggest improvements to the CV for better alignment
  with target roles.

When answering:

1. If the task requires reading the CV, call the CV extraction
   tool before answering.

2. If the task involves evaluating job postings, call the
   job posting tool before answering.

3. Compare and reason about the information before providing
   your final response.

Response format:

- Be clear and concise.
- Use bullet points or numbered lists.
- Use section headers when appropriate.
- Support statements with evidence from the CV or job postings.
- Avoid vague language.

Constraints:

- Do not invent or guess details.
- Only use information available in the CV, job postings,
  or provided context.
- Keep your tone professional, friendly, and supportive.
""")

In [13]:
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

NameError: name 'Image' is not defined

In [21]:
from langgraph.checkpoint.memory import MemorySaver
memory= MemorySaver()
react_graph_memory= builder.compile(checkpointer=memory)

NameError: name 'builder' is not defined

In [22]:
config={"configurable":{"thread_id":"1"}}
result =react_graph.invoke({"messages":messages},config)


NameError: name 'react_graph' is not defined

In [23]:
import os
import requests
from langchain_core.messages import HumanMessage

# 1. Path to your local CV file
cv_path = r"C:\Users\MANIKANTA\OneDrive\Desktop\CV-Template.pdf"

# 2. Extract CV text using our tool
cv_text = extract_cv_text.invoke({"file_path": cv_path})

# 3. Analyze CV using the LLM
prompt = f"""
Analyze the following resume text:

{cv_text}
   | //';ovcx
Please summarize:
1. Candidate Name
2. Education
3. Technical Skills
4. Key Strengths
"""

analysis = llm.invoke([HumanMessage(content=prompt)])
print("--- CV ANALYSIS ---")
print(analysis.content)

# 4. Fetch live Python job listings from Himalayas API
print("\n--- FETCHING JOBS ---")
job_res = requests.get("https://himalayas.app/jobs/api/search?q=python")
if job_res.status_code == 200:
    jobs = job_res.json().get("jobs", [])[:3]
    for idx, job in enumerate(jobs, 1):
        # Extract location safely without triggering an IndexError on empty lists
        locations = job.get('locationRestrictions') or ['Remote']
        location = locations[0] if locations else 'Remote'

        print(f"{idx}. {job.get('title')} at {job.get('companyName')} ({location})")

--- CV ANALYSIS ---
**1. Candidate Name**  
- Fred Smith, CFA  

**2. Education**  
- Chartered Financial Analyst (CFA), 2010‑2014, CFA Institute  
- BSc (Mathematics), First‑Class Honours, ABC University, UK, 2007‑2010  

**3. Technical Skills**  
- Advanced proficiency in Bloomberg, Business Objects, Hyperion, C++  
- Expertise in LBO modelling, financial & commercial due‑diligence, valuation, and financial reporting  
- Strong analytical tools: Excel, portfolio management, and data‑analysis frameworks  

**4. Key Strengths**  
- Proven ability to lead multi‑person teams in high‑stakes private‑equity, investment‑banking, and audit environments  
- Track record of successful deal sourcing, due diligence, and execution across diverse sectors (manufacturing, consumer goods, oil & gas, insurance)  
- Adept at risk identification and mitigation (e.g., uncovered fraudulent investment opportunities)  
- Strategic origination and proposal skills, demonstrated through in‑house cross‑team init

In [24]:
from langchain_core.messages import HumanMessage

def evaluate_job_match(cv_text: str, job_title: str, job_company: str, job_description: str) -> str:
    """
    Compares CV text against a single job description and provides a relevance score and recommendation.
    """
    matching_prompt = f"""
    You are an expert AI Career Coach and Recruiter.

    Candidate Resume:
    {cv_text}

    Target Job Details:
    - Title: {job_title}
    - Company: {job_company}
    - Description: {job_description}

    Evaluate the match between the candidate and this job.
    Provide your output in the following format:

    **Job**: {job_title} at {job_company}
    **Match Score**: [0-100]%
    **Verdict**: [Strong Match / Potential Match / Low Match]
    **Key Matching Skills**: [List 2-3 overlapping skills]
    **Missing Requirements**: [List key gaps]
    **Summary & Recommendation**: [1-2 sentences on whether the candidate should apply and why]
    """

    response = llm.invoke([HumanMessage(content=matching_prompt)])
    return response.content

# Test the matching chain on the first fetched job
if job_res.status_code == 200 and jobs:
    first_job = jobs[0]

    # Extract job description or excerpt safely
    job_desc = first_job.get('description') or first_job.get('excerpt') or "Python development role requiring backend engineering skills."

    match_result = evaluate_job_match(
        cv_text=cv_text,
        job_title=first_job.get('title'),
        job_company=first_job.get('companyName'),
        job_description=job_desc
    )

    print("--- JOB MATCHING EVALUATION ---")
    print(match_result)

--- JOB MATCHING EVALUATION ---
**Job**: Python Developer at Hitapps  
**Match Score**: 18%  
**Verdict**: Low Match  
**Key Matching Skills**: • Advanced knowledge of C++ (programming mindset)  
• Bachelor’s in Mathematics (analytical thinking)  
• Experience with data analysis tools (Bloomberg, Business Objects)  
**Missing Requirements**: • No demonstrated proficiency in Python or web frameworks (FastAPI, Django REST, Flask)  
• No evidence of building backend services or REST APIs  
• Lacks experience with relational databases, queues (Celery, RabbitMQ, Kafka), Redis, Docker, S3, CI/CD, or testing frameworks (pytest)  
• No exposure to AI‑related tools or RAG/LLM integration, which are key for Hitapps’ back‑end AI platform  
**Summary & Recommendation**: The candidate’s background is strongly finance‑centric and does not cover the technical stack required for a Python development role at Hitapps. Unless the candidate has recently gained substantial Python and backend development ex

In [25]:
from langchain_core.messages import HumanMessage

def evaluate_job_match(cv_text: str, job_title: str, job_company: str, job_description: str) -> str:
    """
    Compares CV text against a single job description and provides a relevance score and recommendation.
    """
    matching_prompt = f"""
    You are an expert AI Career Coach and Recruiter.

    Candidate Resume:
    {cv_text}

    Target Job Details:
    - Title: {job_title}
    - Company: {job_company}
    - Description: {job_description}

    Evaluate the match between the candidate and this job.
    Provide your output in the following format:

    **Job**: {job_title} at {job_company}
    **Match Score**: [0-100]%
    **Verdict**: [Strong Match / Potential Match / Low Match]
    **Key Matching Skills**: [List 2-3 overlapping skills]
    **Missing Requirements**: [List key gaps]
    **Summary & Recommendation**: [1-2 sentences on whether the candidate should apply and why]
    """

    response = llm.invoke([HumanMessage(content=matching_prompt)])
    return response.content

# Test the matching chain on the first fetched job
if job_res.status_code == 200 and jobs:
    first_job = jobs[0]

    # Extract job description or excerpt safely
    job_desc = first_job.get('description') or first_job.get('excerpt') or "Python development role requiring backend engineering skills."

    match_result = evaluate_job_match(
        cv_text=cv_text,
        job_title=first_job.get('title'),
        job_company=first_job.get('companyName'),
        job_description=job_desc
    )

    print("--- JOB MATCHING EVALUATION ---")
    print(match_result)

--- JOB MATCHING EVALUATION ---
**Job**: Python Developer at Hitapps  
**Match Score**: 10%  
**Verdict**: Low Match  
**Key Matching Skills**: None (C++ programming experience only)  
**Missing Requirements**: Proficiency in Python; experience with REST‑API frameworks (FastAPI/Django/Flask); relational database expertise (PostgreSQL/MySQL); background‑task/queue systems (Celery, RabbitMQ, Kafka); Docker & S3; testing and CI/CD culture; LLM integration and related AI tooling.  
**Summary & Recommendation**: Fred Smith’s profile is heavily finance‑oriented with no demonstrated Python or backend‑development experience. The skill gap is too large for the Hitapps Python Developer role; he should not apply in its current form.
